In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from carga_cammesa import union_filtro
from carga_cammesa import meses_faltantes
from carga_cammesa import apilar_ediciones
from carga_cammesa import detectar_revisados
from carga_cammesa import deduplicar
from carga_cammesa import csv_tabla

## 1. Períodos faltantes

In [3]:
rutas = sorted(Path("../data/raw/cammesa").glob("CAMMESA_MEM_ReqMax_*.xlsx"))
tabla = union_filtro(rutas)

In [4]:
tabla_hueco = tabla.drop(index=10)

In [5]:
ruta_temp = Path("../data/processed/OPENMETEO_TRELEW_TempMensual.csv")
temperatura = pd.read_csv(ruta_temp, parse_dates=["MES"])
meses_faltantes(temperatura)

DatetimeIndex([], dtype='datetime64[us]', freq='MS')

In [6]:
print("Tabla completa: ",meses_faltantes(tabla))
print("Tabla con faltantes: ",meses_faltantes(tabla_hueco))

Tabla completa:  DatetimeIndex([], dtype='datetime64[us]', freq='MS')
Tabla con faltantes:  DatetimeIndex(['2019-11-01'], dtype='datetime64[us]', freq='MS')


## 2. Ediciones y meses revisados

In [7]:
ediciones = apilar_ediciones(rutas)
print(len(ediciones))

141


In [8]:
revisados = detectar_revisados(ediciones)
print(sum(revisados))

24


In [9]:
serie = deduplicar(ediciones)
len(serie)

93

In [10]:
serie["EDICION"].value_counts()

EDICION
2026-09    45
2022-12    24
2024-12    24
Name: count, dtype: int64

## 3. Serie final

In [11]:
tabla.info()
tabla["REVISADO"].sum()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   MES                    93 non-null     datetime64[us]
 1   MAX_REAL_MW            93 non-null     float64       
 2   POTENCIA_DECLARADA_MW  93 non-null     float64       
 3   EDICION                93 non-null     str           
 4   DESCARGA               93 non-null     str           
 5   REVISADO               93 non-null     bool          
dtypes: bool(1), datetime64[us](1), float64(2), str(2)
memory usage: 5.4 KB


np.int64(24)

In [12]:
ruta_salida = Path("../data/processed/CAMMESA_CTRELEUW_ReqMax.csv")
csv_tabla(tabla, ruta_salida)